# Zambia (2014) — MODIS / FIRMS exploration v1

**Goal:** Work with **MODIS-derived fire data** for Zambia for **2013-08-04 → 2014-08-03** (12 months before survey start), using **FIRMS bulk / archive downloads** and/or a small **`api/area`** test when **`FIRMS_MAP_KEY`** is set.

**Survey anchor (VACS Zambia 2014):** fieldwork **2014-08-04 → 2014-10-05**.

**Exposure window (v1):** **2013-08-04** through **2014-08-03** (inclusive).

**Terminology:** **MODIS** = instrument on **Terra/Aqua**. **Hotspot CSVs** (FIRMS) and **burned-area grids** (e.g. **MCD64A1**) are different derived products.

**Pipeline:** **§2b** optional **`api/area`** smoke test → **§3–§4** bulk files in **`data/raw/exposure_firms/zambia/`** → **§5** optional availability / country smoke tests → **§6** burned-area pointers → **§7** optional **Earth Engine** Hansen admin-2 demo (requires EE auth; see `gee_zambia/README.md`).

## §1 — Repo paths, bulk input folder, FIRMS country code

- **`BULK_DIR`** — put FIRMS-exported **CSV / TXT / shapefile** (or unzip here) from **[Archive Download](https://firms.modaps.eosdis.nasa.gov/download/)** (or related FIRMS export). Create the folder if it does not exist.
- **`FIRMS_CC`:** FIRMS uses a **3-letter country id** for Zambia; this matches **ISO 3166-1 alpha-3 `ZMB`**. If a future export uses a different label, change this constant.
- **`FIRMS_MAP_KEY`** (optional): still read from repo **`.env`** for optional API cells in §5 — not required for bulk ingest.

In [ ]:
from __future__ import annotations

import os
import sys
from datetime import date
from pathlib import Path

import pandas as pd
from IPython.display import display

_repo = next(
    (
        d
        for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (d / "utils" / "repo_paths.py").is_file() and (d / "data" / "raw").is_dir()
    ),
    None,
)
if _repo is None:
    raise RuntimeError("Cannot find repo root (expected utils/repo_paths.py and data/raw/).")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from utils.repo_paths import find_repo_root


def load_map_key(repo_root: Path) -> str:
    env_path = repo_root / ".env"
    if env_path.is_file():
        for line in env_path.read_text().splitlines():
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if line.upper().startswith("FIRMS_MAP_KEY="):
                return line.split("=", 1)[1].strip().strip('"').strip("'")
    return os.environ.get("FIRMS_MAP_KEY", "").strip()


ROOT = find_repo_root()
MAP_KEY = load_map_key(ROOT)

BULK_DIR = ROOT / "data" / "raw" / "exposure_firms" / "zambia"
BULK_DIR.mkdir(parents=True, exist_ok=True)

ISO3 = "ZMB"
FIRMS_CC = "ZMB"  # FIRMS country id for Zambia (matches ISO3; adjust if export docs differ)

print("ROOT =", ROOT)
print("BULK_DIR =", BULK_DIR)
print("FIRMS_CC =", FIRMS_CC)
print("FIRMS_MAP_KEY:", "loaded" if MAP_KEY else "not set (OK for bulk-only)")

## §2 — Zambia 2014 windows

Edit if the PI changes the exposure definition.

In [ ]:
SURVEY_FIELD_START = date(2014, 8, 4)
SURVEY_FIELD_END = date(2014, 10, 5)
EXPOSURE_START = date(2013, 8, 4)
EXPOSURE_END = date(2014, 8, 3)

windows = pd.DataFrame(
    [
        {"label": "survey_field", "start": SURVEY_FIELD_START, "end": SURVEY_FIELD_END},
        {"label": "exposure_12mo_pre_survey", "start": EXPOSURE_START, "end": EXPOSURE_END},
    ]
)
display(windows)

## §2b — Quick FIRMS `api/area` test (Zambia box, MODIS_SP)

**BBox (lon/lat):** `21.9990553,-18.0762145,33.7088556,-8.2749338` (OSM Zambia).

The next cell asks for **10 days** of data, then **5 days** for the same start date, so you can see whether **`DAY_RANGE` > 5** is accepted. NASA’s [area API](https://firms.modaps.eosdis.nasa.gov/api/area/) doc says **`DAY_RANGE` is 1–5** — expect the 10-day call to **fail** with HTTP 400.

In [ ]:
import urllib.error

# Whole Zambia: min_lon, min_lat, max_lon, max_lat (OSM / Nominatim)
bbox = "21.9990553,-18.0762145,33.7088556,-8.2749338"
source = "MODIS_SP"
day0 = EXPOSURE_START  # first day of the 5- or 10-day window

if not MAP_KEY:
    print("Add FIRMS_MAP_KEY to repo .env to run this test.")
else:
    base = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{MAP_KEY}/{source}/{bbox}"

    # Does DAY_RANGE > 5 work? (NASA docs say max is 5 — expect 400 for 10)
    url10 = f"{base}/10/{day0.isoformat()}"
    try:
        df10 = pd.read_csv(url10)
        print("10-day request: OK, rows =", len(df10))
        display(df10.head(3))
    except urllib.error.HTTPError as e:
        print("10-day request:", e)

    # Allowed case (compare row count / dates)
    url5 = f"{base}/5/{day0.isoformat()}"
    df5 = pd.read_csv(url5)
    print("5-day request: OK, rows =", len(df5))
    display(df5.head(3))

## §3 — How to obtain bulk FIRMS MODIS data (no `/api/countries`)

1. Open **[FIRMS Archive Download](https://firms.modaps.eosdis.nasa.gov/download/)**.
2. Choose **MODIS** (C6 or as offered), set **country / region** to **Zambia** and your **date range** `[EXPOSURE_START, EXPOSURE_END]` (or full months covering that span).
3. Submit the request; when you receive the email, download **CSV / shapefile / GeoJSON** as provided.
4. Copy or unzip into **`BULK_DIR`** above (`data/raw/exposure_firms/zambia/`).

Alternate reference: **country yearly summary** pages (if useful for sanity checks): [FIRMS country yearly summary](https://firms2.modaps.eosdis.nasa.gov/country) — still web UI, not the broken list API.

For **burned-area grids** (not hotspot points), use **LP DAAC MCD64A1** or **Google Earth Engine** — see §6.

## §4 — Ingest local FIRMS CSV/TXT from `BULK_DIR`

FIRMS hotspot files are usually comma-separated with columns like **`latitude`**, **`longitude`**, **`acq_date`**, **`acq_time`**, **`satellite`**, **`instrument`**, **`confidence`** (names can vary slightly by export).

This cell loads **all** `*.csv` and `*.txt` in `BULK_DIR`, concatenates, and filters rows into the exposure window when a parseable acquisition date is found.

In [ ]:
paths = sorted(BULK_DIR.glob("*.csv")) + sorted(BULK_DIR.glob("*.txt"))
if not paths:
    print("No .csv or .txt in BULK_DIR yet. Add FIRMS export files, then re-run.")
    fires_raw = pd.DataFrame()
else:
    frames = []
    for p in paths:
        try:
            df = pd.read_csv(p, low_memory=False)
        except UnicodeDecodeError:
            df = pd.read_csv(p, low_memory=False, encoding="latin1")
        df["_source_file"] = p.name
        frames.append(df)
    fires_raw = pd.concat(frames, ignore_index=True)
    print("Loaded", len(paths), "file(s);", fires_raw.shape[0], "rows")

if not fires_raw.empty:
    display(fires_raw.head(3))
    print("Columns:", list(fires_raw.columns))

    date_col = next(
        (c for c in fires_raw.columns if c.lower() in ("acq_date", "acquisition_date", "date")),
        None,
    )
    if date_col is None:
        print("WARN: no acq_date-like column; skip date filter or rename column.")
        fires_exposure = fires_raw
    else:
        d = pd.to_datetime(fires_raw[date_col], errors="coerce")
        m = d.dt.date.between(EXPOSURE_START, EXPOSURE_END)
        fires_exposure = fires_raw.loc[m].copy()
        fires_exposure["_acq_date_parsed"] = d.loc[m].dt.date
        print(
            f"Rows in exposure window [{EXPOSURE_START} .. {EXPOSURE_END}]:",
            len(fires_exposure),
        )
        display(fires_exposure.head())

## §5 — (Optional) Live API checks — only if FIRMS is up

Skip this section when APIs are flaky. **`data_availability`** uses your **`FIRMS_MAP_KEY`** and can confirm archive end dates vs your exposure window.

In [ ]:
if not MAP_KEY:
    print("Skipping API check (no FIRMS_MAP_KEY).")
else:
    try:
        da_url = f"https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/{MAP_KEY}/all"
        avail = pd.read_csv(da_url)
        display(avail)
    except Exception as e:
        print("data_availability request failed:", type(e).__name__, e)

    # Tiny country smoke test (does not replace bulk archive for a full year)
    try:
        country_url = f"https://firms.modaps.eosdis.nasa.gov/api/country/csv/{MAP_KEY}/MODIS_SP/{FIRMS_CC}/5"
        test = pd.read_csv(country_url)
        print("api/country smoke test OK, rows:", len(test))
    except Exception as e:
        print("api/country smoke test skipped/failed:", type(e).__name__, e)

## §6 — Burned area (MODIS-derived) — bulk / science products

**Hotspot CSVs** ≠ **burned-area fraction**. For burned area over the same window:

- **LP DAAC / Earthdata:** search **MCD64A1** (monthly burned area).
- **Google Earth Engine:** e.g. `MODIS/006/MCD64A1` (verify current collection ID in the Code Editor).

Download HDF/GeoTIFF to e.g. `data/raw/exposure_modis_ba/zambia/` and document the path in your weekly PI note.

## §7 — Earth Engine demo (Hansen forest loss, Zambia admin-2)

**Google Earth Engine (GEE)** runs raster analysis in the cloud over catalog layers. This demo uses **Hansen Global Forest Change** ([`UMD/hansen/global_forest_change_2025_v1_13`](https://developers.google.com/earth-engine/datasets/catalog/UMD_hansen_global_forest_change_2025_v1_13)): band **`lossyear`** encodes **calendar-year** loss (2001→1 … 2025→25 in v1.13), not sub-monthly dates. **September 2013** falls inside your exposure window for context; the CSV is **2013 annual** Hansen loss area per **FAO GAUL 2015** admin-2 polygon, not “September-only” loss. Monthly **fire** from MODIS/VIIRS in GEE would be a separate step.

**Export CSV (after EE auth):** from repo root, `python -m gee_zambia.hansen_zonal --year 2013` — instructions in [`gee_zambia/README.md`](../gee_zambia/README.md).

In [ ]:
# Load GEE script output (run §1 first so ROOT exists)
from IPython.display import display

gee_csv = ROOT / "data" / "raw" / "exposure_gee" / "zambia" / "hansen_loss_y2013_admin2_zambia.csv"
if not gee_csv.is_file():
    print("No GEE output yet. After Earth Engine auth, run from repo root:")
    print("  python -m gee_zambia.hansen_zonal --year 2013")
    print("Expected file:", gee_csv)
else:
    hansen_adm2 = pd.read_csv(gee_csv)
    print("Rows (admin-2 units):", len(hansen_adm2))
    if "loss_area_ha" in hansen_adm2.columns:
        print("Total Hansen 2013 forest loss (ha, summed over units):", round(hansen_adm2["loss_area_ha"].sum(), 2))
    display(hansen_adm2.head(10))